# Cirq interoperability with native registers

Adapted from Qualtran v0.7.0, Apache-2.0.

In [ ]:
import cirq
import numpy as np
from cirq.contrib.svg import SVGCircuit
from IPython.display import display
from flagquantum.circuit import Circuit
from flagquantum.core.ir import CircuitIR, Instruction
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.core.ir.program.registers import Register, RegisterSide
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.ecosystem.cirq import export_cirq, import_cirq
from flagquantum.ecosystem.cirq._register_export import export_register_cirq

## Importing Cirq circuits

The optional adapter converts Cirq circuits into native IR. Its report describes operations and metadata outside the static contract.

### A five-qubit QFT boundary

Cirq can represent the composite Fourier transform as a single gate. Check the adapter diagnostic for that gate.

In [ ]:
qft = cirq.QuantumFourierTransformGate(num_qubits=5)
qft_circuit = cirq.Circuit(qft.on(*cirq.LineQubit.range(5)))
display(cirq.qid_shape(qft))

In [ ]:
qft_report = import_cirq(qft_circuit, allow_lossy=True).report
assert "unsupported_operation" in {issue.code for issue in qft_report.issues}
display(qft_report.to_dict())

### Convert the seeded example

The original random circuit contains an iSWAP. Use its exact decomposition before importing, and compare both matrices.

In [ ]:
qubits = cirq.LineQubit.range(4)
original = cirq.testing.random_circuit(qubits, n_moments=5, op_density=1.0, random_state=52)
lowered = cirq.Circuit()
for op in original.all_operations():
    if op.gate == cirq.ISWAP:
        a, b = op.qubits
        lowered.append([cirq.S(a), cirq.S(b), cirq.CZ(a, b), cirq.SWAP(a, b)])
    else:
        lowered.append(op)
imported = import_cirq(lowered, allow_lossy=True)
assert {issue.code for issue in imported.report.issues} <= {"moment_structure_flattened"}
display(SVGCircuit(original))

### Check the full unitary

In [ ]:
exported = export_cirq(imported.ir)
np.testing.assert_allclose(original.unitary(qubit_order=qubits), exported.circuit.unitary(qubit_order=qubits), atol=2e-12, rtol=0)

### Export the imported IR on explicit qubits

In [ ]:
word = Signature((Register("qubits", opaque_register_type(1), shape=(4,)),))
roundtrip, ports = export_register_cirq(
    imported.ir, word, {"qubits": (0, 1, 2, 3)}, input_quregs={"qubits": [[q] for q in qubits]}
)
display(SVGCircuit(roundtrip.circuit))

In [ ]:
display(original == roundtrip.circuit)

In [ ]:
np.testing.assert_allclose(original.unitary(qubit_order=qubits), roundtrip.circuit.unitary(qubit_order=qubits), atol=2e-12, rtol=0)
display(imported.report.to_dict())

## Exporting native register operations

### Bind an exchange to named wires

In [ ]:
from examples.algorithms.binary_field_arithmetic import FieldOperation

swap_body = (Instruction("swap", (0, 1)),)
swap = FieldOperation(2, swap_body)
pair = Signature.build(x=1, y=1)
binding = {"x": (0,), "y": (1,)}
program = Circuit(2)
program.append(swap)

In [ ]:
external, output = export_register_cirq(
    program, pair, binding, input_quregs={"x": cirq.NamedQubit("q1"), "y": cirq.NamedQubit("q2")}
)
display(SVGCircuit(external.circuit))

### Bind a composite circuit

In [ ]:
named, named_outputs = export_register_cirq(program, pair, binding)
display(named_outputs)

In [ ]:
program.append(swap)
twice, _ = export_register_cirq(program, pair, binding)
np.testing.assert_allclose(twice.circuit.unitary(), np.eye(4), atol=2e-12, rtol=0)
display(SVGCircuit(twice.circuit))

### Inspect a word exchange

The native compilation emits the component gates.

In [ ]:
from flagquantum.compiler.registers.primitives.swap import swap_register_ir

In [ ]:
left, right = (0, 1, 2), (3, 4, 5)
signature = Signature.build(x=3, y=3)
wide = Circuit(6)
wide.append(FieldOperation(6, swap_register_ir(left, right)))

Place the two words on distant external line ranges.

In [ ]:
placed, wide_ports = export_register_cirq(
    wide, signature, {"x": left, "y": right},
    input_quregs={"x": cirq.LineQubit.range(3), "y": cirq.LineQubit.range(100, 103)},
)

Inspect the emitted leaf operations and their operand order.

In [ ]:
ops = list(placed.circuit.all_operations())
expected_pairs = list(zip(wide_ports["x"], wide_ports["y"]))
assert [op.qubits for op in ops] == expected_pairs
display(ops)

In [ ]:
display(SVGCircuit(placed.circuit))

### Left, right and through ports

In [ ]:
from flagquantum.compiler.registers.control.multi_control import multi_and_ir

left_right = Signature((
    Register("ctrl", opaque_register_type(1)),
    Register("result", opaque_register_type(1), side=RegisterSide.RIGHT),
))
fresh = CircuitIR(2, (Instruction("cx", (0, 1)),))

Supply only the input register arrays. A right-only port is placed on fresh external qubits.

In [ ]:
control = cirq.NamedQubit("input")
fresh_output, fresh_ports = export_register_cirq(fresh, left_right, {"ctrl": (0,), "result": (1,)}, input_quregs={"ctrl": control})
assert fresh_ports["ctrl"][0] == control
assert fresh_ports["result"][0] != control

In [ ]:
display(SVGCircuit(fresh_output.circuit))

The original four-control MultiAnd has right-only junk and target ports. Its literal lowering uses inverse phase gates beyond the current Cirq contract. The placement example above uses a primitive already in that contract.

In [ ]:
ladder = CircuitIR(7, multi_and_ir((0, 1, 2, 3), (1, 1, 1, 1), (4, 5, 6)))
ladder_report = export_cirq(ladder, allow_lossy=True).report
assert {issue.operation_name for issue in ladder_report.issues} <= {"tdg", "sdg"} and not ladder_report.lossless
display(ladder_report.to_dict())

## Modular exponentiation export boundary

Retain the original large instance's signature and call graph. Inspect the export diagnostic on a smaller compiled instance with measurement feedback.

In [ ]:
from flagquantum.algorithms.primitives.modular_exponentiation import ModularExponentiation
from flagquantum.dynamic import DynamicCircuit

large = ModularExponentiation(8, 13*17, 32, 32)
display(large.source_signature)
display(large.source_call_graph().direct_calls)
assert large.classical_values(13) == (13, pow(8, 13, 221))

In [ ]:
modexp = ModularExponentiation(2, 3, 2, 2)
work, slots = modexp.workspace_requirements()
native = DynamicCircuit(4 + work)
native.append(modexp.make_on(exponent=(0, 1), target=(2, 3),
    workspace=tuple(range(4, 4 + work)), classical_bits=tuple(range(slots))))

In [ ]:
modexp_report = export_cirq(native, allow_lossy=True).report
assert not modexp_report.lossless
display(sorted({issue.code for issue in modexp_report.issues}))